In [1]:
import gymnasium as gym

import numpy as np
import time
import matplotlib
import matplotlib.pyplot as plt
import random 
import numpy as np
from collections import defaultdict
#import tensorflow as tf
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Input
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from collections import deque
 
 

env=gym.make('CartPole-v1')#,render_mode='human')

(state,_)=env.reset()

In [2]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),  # Increased size for hidden layers
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, states):
        input_list = np.array(states,dtype=np.float32)
        x=torch.from_numpy(input_list).float()
        return self.fc(x)
    

In [3]:
class DQAgent:

    def __init__(self, state_dim, action_dim, gamma=0.99, lr=0.001, batch_size=64, buffer_size=10000):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.memory = deque(maxlen=buffer_size)

        # Models
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.update_target_model()

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.loss_fn = nn.SmoothL1Loss() 
        
    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q_values = self.model(state_tensor)
            return torch.argmax(q_values).item()
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
            
    def train(self): 
        
        if len(self.memory) < self.batch_size:
            return  # Not enough samples to train

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)

        # Compute target Q-values
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            max_next_q_values = next_q_values.max(dim=1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        # Compute current Q-values
        q_values = self.model(states)
        current_q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Compute loss
        loss = self.loss_fn(current_q_values, target_q_values)

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        

In [4]:
def treatState(state,level,goal):
    tmp_level= min(level/500,1)
    state =np.append(state,tmp_level)
    return state

In [5]:

def KL(a,b):
       
        if len(a)<len(b):
            return 1
        return np.sum(a[-100:]*np.log(a[-100:]/b))


In [6]:
def changeReward(reward,done,num,goal):
    return reward
    if done:
        if num>goal:
             return reward-0.01*(num-goal)#+10
        return reward-0.01*(goal-num)#-10
    else:
        if num>goal:
             return -(reward+0.01*(num-goal))
        return 1 + 0.01 * num

## test

In [7]:
episodeNumber=30000
timeSteps=1000
episodesResults=[]
episodesTestResults=[]
episodesTestResultsDeviation=[]
goaldistr=np.random.normal(loc=100,scale=3,size=100)
numberOfTest=50
agent = DQAgent(5, 2,batch_size=512)
goal=130


In [8]:
for episodeIndex in range(1,episodeNumber+1):
    state,_=env.reset()
    state=treatState(state,0,goal)
    
    
   
    for timeIndex in range(timeSteps+1):
        action=agent.get_action(state)
        next_state, reward, done, _, _ = env.step(action)
        next_state=treatState(next_state,timeIndex+1,goal)
        agent.store_transition(state, action, changeReward(reward,done,timeIndex,goal), next_state, done)
        #agent.store_transition(state, action, KL(np.append(episodesResults,timeIndex),goaldistr), next_state, done)
        state=next_state
        agent.train()
        if (done or timeIndex==timeSteps-1 ):
            
            episodesResults.append(timeIndex)
            break
    
    if(episodeIndex %100 ==0):
        agent.update_target_model()
    if(episodeIndex %250 ==0):
        print("episode : ",episodeIndex)
    if(episodeIndex %500 ==0):
        tmp_epsilon=agent.epsilon
        agent.epsilon=0
        testResultList=[]
        for episodeIndex in range(numberOfTest):
            state,_=env.reset()
            state=treatState(state,0,goal)
            for timeIndex in range(timeSteps):
                action=agent.get_action(tuple(state))
                next_state, reward, done, _, _ = env.step(action)
                next_state=treatState(next_state,timeIndex+1,goal)
                state=next_state
                if (done):
                    testResultList.append(timeIndex)
                    break
        episodesTestResults.append(sum(testResultList)/numberOfTest)
        episodesTestResultsDeviation.append(np.std(episodesTestResults))
        agent.epsilon=tmp_epsilon

env.close()

C:\Users\mathi\AppData\Local\Temp\ipykernel_26792\2635204200.py:43: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  states = torch.FloatTensor(states)


episode :  250
episode :  500
episode :  750
episode :  1000
episode :  1250
episode :  1500
episode :  1750
episode :  2000
episode :  2250
episode :  2500
episode :  2750
episode :  3000
episode :  3250
episode :  3500
episode :  3750
episode :  4000
episode :  4250
episode :  4500
episode :  4750
episode :  5000
episode :  5250
episode :  5500
episode :  5750
episode :  6000
episode :  6250
episode :  6500
episode :  6750
episode :  7000
episode :  7250
episode :  7500
episode :  7750
episode :  8000
episode :  8250
episode :  8500
episode :  8750
episode :  9000
episode :  9250
episode :  9500
episode :  9750
episode :  10000
episode :  10250
episode :  10500
episode :  10750
episode :  11000
episode :  11250
episode :  11500
episode :  11750
episode :  12000
episode :  12250
episode :  12500
episode :  12750
episode :  13000
episode :  13250
episode :  13500
episode :  13750
episode :  14000
episode :  14250
episode :  14500
episode :  14750
episode :  15000
episode :  15250
episo

In [9]:
goal10LearnerEp=episodesResults
%store goal10LearnerEp

DQN3episodesTestResults=episodesTestResults
%store DQN3episodesTestResults

DQN3episodesTestResultsDeviation=episodesTestResultsDeviation
%store DQN3episodesTestResultsDeviation

Stored 'goal10LearnerEp' (list)
Stored 'DQN3episodesTestResults' (list)
Stored 'DQN3episodesTestResultsDeviation' (list)


In [10]:
print(episodesTestResultsDeviation)
print(episodesTestResults)
print(len(episodesResults))

[0.0, 13.329999999999998, 39.410064140464875, 152.0933768281841, 146.64811325073364, 141.07011609676783, 130.72574994223282, 124.8452638268669, 139.68595984485262, 135.3585536122487, 131.12984639232988, 125.7271045094626, 121.8226383874058, 124.98706845763147, 120.75829913048996, 120.98581342454992, 117.37363809823408, 118.00536062316182, 116.88556113659234, 115.36227048303098, 114.18841187220222, 112.978097987147, 112.52510781866874, 112.04351207107987, 111.22867306877308, 125.75995729867006, 125.9886827543147, 124.23242054425937, 123.7638679261553, 123.18643313648177, 122.64480253803735, 121.79304620024239, 120.44117710456177, 119.2315648623114, 117.5821461027245, 120.33001930903852, 122.28700237988977, 122.01399405685551, 124.34572877279226, 124.23786221498663, 122.84554698691838, 122.18983270067727, 121.42348122570516, 121.11977412158785, 120.02159162458547, 119.25781072651664, 118.07456689787738, 116.94525074007805, 115.9693479558548, 114.85758873944725, 113.95236763245718, 113.46